In [1]:
from mmpose.apis.inference import inference_topdown
from mmpose.apis import init_model as init_pose_estimator
from mmpose.structures import merge_data_samples
from mmpose.registry import VISUALIZERS

import cv2
import mmcv
import numpy as np
import os
import time

from helpers.definitions import *

c:\Users\fisv\AppData\Local\miniconda3\envs\openpose\Lib\site-packages\mmengine\optim\optimizer\zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


In [ ]:
# Specify the frame to be loaded
start_frame = 600
end_frame = 1000
cam_name = "gopro1" # start camera
cam_idx = cam_names.index(cam_name)

show = False
draw_bbox = True
kpt_thresh = 0.2

if end_frame is None:
    suffix = 'png'
else:
    suffix = 'mp4'

input_file = f"npz/updated_{cam_name}_bbox.npz"
output_file = f"vis_dir/updated_{cam_name}_pose.{suffix}"

In [ ]:
def process_one_image(frame_idx,
                      img,
                      bboxes,
                      pose_estimator,
                      visualizer=None,
                      show_interval=0):
    """Visualize predicted keypoints of one image."""

    # predict keypoints
    pose_results = inference_topdown(pose_estimator, img, bboxes)
    data_samples = merge_data_samples(pose_results)

    # show the results
    if isinstance(img, str):
        img = mmcv.imread(img, channel_order='rgb')
    elif isinstance(img, np.ndarray):
        img = mmcv.bgr2rgb(img)

    if visualizer is not None:
        visualizer.add_datasample(
            'result',
            img,
            data_sample=data_samples,
            draw_gt=False,
            draw_heatmap=False,
            draw_bbox=draw_bbox,
            show_kpt_idx=False,
            skeleton_style='mmpose',
            show=show,
            wait_time=show_interval,
            kpt_thr=kpt_thresh)

    # if there is no instance detected, return None
    return data_samples.get('pred_instances', None)


In [4]:
# build pose estimator
pose_estimator = init_pose_estimator(
    'configs/hand_2d_keypoint/rtmpose/hand5/rtmpose-m_8xb256-210e_hand5-256x256.py',
    'checkpoints/hands/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth',
    device='cuda',
    cfg_options=dict(
        model=dict(test_cfg=dict(output_heatmaps=False))))

# build visualizer
pose_estimator.cfg.visualizer.radius = 3
pose_estimator.cfg.visualizer.alpha = 0.8
pose_estimator.cfg.visualizer.line_width = 1
visualizer = VISUALIZERS.build(pose_estimator.cfg.visualizer)
# the dataset_meta is loaded from the checkpoint and
# then pass to the model in init_pose_estimator
visualizer.set_dataset_meta(
    pose_estimator.dataset_meta, skeleton_style='mmpose')

Loads checkpoint by local backend from path: checkpoints/hands/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth


c:\Users\fisv\AppData\Local\miniconda3\envs\openpose\Lib\site-packages\mmengine\runner\checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.lo

In [5]:
bboxes = np.load('npz/updated_gopro1_bbox.npz')['bbox']

resolution = (1920, 1080)

poses_path = 'pose_outputs'
npz_path = 'npz'
video_path = 'videos'
num_frames = 5400

video_dirs = []
for cam_name in cam_names:
    video_dirs.append(os.path.join(video_path, f'{cam_name}_selected_mmpose.mp4'))

frame_idx = start_frame # start frame
if end_frame is None:

    # Load the given frame
    video = cv2.VideoCapture(video_dirs[cam_idx])  # Replace with your image path
    if not video.isOpened():
            raise FileNotFoundError(f"Cannot open video file: {video_dirs[cam_idx]}")

    # Set the video to the nth frame
    video.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    # Read the frame
    success, frame = video.read()
    if not success:
        raise ValueError(f"Frame {frame_idx} does not exist in the video.")
    
    pred_instances = process_one_image(frame_idx, frame, [bboxes[frame_idx]], pose_estimator, visualizer)
    img_vis = visualizer.get_image()
    mmcv.imwrite(img_vis, output_file)

    video.release()

else:

    # Load the given frame
    cap = cv2.VideoCapture(video_dirs[cam_idx])  # Replace with your image path
    if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video file: {video_dirs[cam_idx]}")

    video_writer = None
    pred_instances_list = []
    frame_idx = start_frame

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    while cap.isOpened() and frame_idx < end_frame:
        success, frame = cap.read()
        frame_idx += 1

        if not success:
            break

        # topdown pose estimation
        pred_instances = process_one_image(frame_idx, frame, [bboxes[frame_idx]], pose_estimator, visualizer, 0.001)

        # output videos
        frame_vis = visualizer.get_image()

        if video_writer is None:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            # the size of the image with visualization may vary
            # depending on the presence of heatmaps
            video_writer = cv2.VideoWriter(
                output_file,
                fourcc,
                30,  # saved fps
                (frame_vis.shape[1], frame_vis.shape[0]))

        video_writer.write(mmcv.rgb2bgr(frame_vis))

        if show:
            # press ESC to exit
            if cv2.waitKey(5) & 0xFF == 27:
                break

            time.sleep(0)

    if video_writer:
        video_writer.release()

    cap.release()
    

c:\Users\fisv\AppData\Local\miniconda3\envs\openpose\Lib\site-packages\mmdet\models\layers\se_layer.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
c:\Users\fisv\AppData\Local\miniconda3\envs\openpose\Lib\site-packages\mmdet\models\backbones\csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
